# Public Submission Notebook

This notebook downloads code + model checkpoint from GitHub and builds `submission.csv` for Kaggle.

## 1) Configure links

Replace placeholders below before running.

In [ ]:
REPO_ZIP_URL = "https://github.com/KekStroke/spoken-numbers-recognition/archive/refs/tags/v1.0.0.zip"
CHECKPOINT_URL = "https://github.com/KekStroke/spoken-numbers-recognition/releases/download/v1.0.0/best.pt"

RAW_DATA_ROOT = "/kaggle/input/asr-2026-spoken-numbers-recognition-challenge"
PROCESSED_DATA_ROOT = "/kaggle/working/data_processed_16k_test"
SAMPLE_SUBMISSION = f"{RAW_DATA_ROOT}/sample_submission.csv"
OUTPUT_CSV = "/kaggle/working/submission.csv"

BATCH_SIZE = 32
NUM_WORKERS = 2


## 2) Install dependencies and fetch repository

In [ ]:
import os
import pathlib
import subprocess
import sys
import zipfile
from urllib.request import urlretrieve

subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "torch",
    "pandas",
    "numpy",
    "librosa",
    "scipy",
    "tqdm",
    "soundfile",
])

repo_zip_path = "/kaggle/working/repo.zip"
urlretrieve(REPO_ZIP_URL, repo_zip_path)

with zipfile.ZipFile(repo_zip_path, "r") as zf:
    zf.extractall("/kaggle/working")

repo_candidates = sorted(pathlib.Path("/kaggle/working").glob("*-*"))
repo_dir = None
for candidate in repo_candidates:
    if (candidate / "src" / "make_submission.py").exists():
        repo_dir = candidate
        break

if repo_dir is None:
    raise RuntimeError("Could not locate extracted repository folder with src/make_submission.py")

os.chdir(str(repo_dir))
print(f"Using repository directory: {repo_dir}")


## 3) Download checkpoint from GitHub Release

In [ ]:
checkpoint_path = "/kaggle/working/best.pt"
urlretrieve(CHECKPOINT_URL, checkpoint_path)
print(f"Checkpoint saved to: {checkpoint_path}")


## 4) Prepare test audio (mono WAV 16k)

In [ ]:
prep_cmd = [
    sys.executable,
    "src/dataset/preprocess_audio.py",
    "--data-root",
    RAW_DATA_ROOT,
    "--output-dir",
    PROCESSED_DATA_ROOT,
    "--splits",
    "test",
    "--clip-splits",
    "",
    "--copy-csv",
    "--overwrite",
]

print("Running:", " ".join(prep_cmd))
subprocess.check_call(prep_cmd)


## 5) Build submission.csv

In [ ]:
cmd = [
    sys.executable,
    "-m",
    "src.make_submission",
    "--checkpoint",
    checkpoint_path,
    "--data-root",
    PROCESSED_DATA_ROOT,
    "--sample-submission",
    SAMPLE_SUBMISSION,
    "--output",
    OUTPUT_CSV,
    "--batch-size",
    str(BATCH_SIZE),
    "--num-workers",
    str(NUM_WORKERS),
    "--device",
    "auto",
]

print("Running:", " ".join(cmd))
subprocess.check_call(cmd)


## 6) Validate and preview output

In [ ]:
import pandas as pd

submission_df = pd.read_csv(OUTPUT_CSV)
sample_df = pd.read_csv(SAMPLE_SUBMISSION)

print("Submission shape:", submission_df.shape)
print("Sample shape:", sample_df.shape)
print("Columns:", submission_df.columns.tolist())
display(submission_df.head())

assert list(submission_df.columns) == ["filename", "transcription"], "Unexpected submission columns"
assert len(submission_df) == len(sample_df), "Row count mismatch with sample submission"
assert submission_df["transcription"].notna().all(), "Found empty transcriptions"

print(f"Ready to submit: {OUTPUT_CSV}")
